Training of mentions extractor:

In [ ]:
import mentions_extractor_trainer
import torch
from transformers import AutoModel, AutoTokenizer

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
encoder = AutoModel.from_pretrained("SpanBERT/spanbert-base-cased").to(device)
tokenizer = AutoTokenizer.from_pretrained('bert-base-cased')
runner = mentions_extractor_trainer.MentionsExtractorTrainer(encoder, tokenizer, 1024)
runner.train()

Forward pass in linker:

In [1]:
import torch
import utils
from transformers import AutoModel, AutoTokenizer
import mentions_extractor
import mentions_linker

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
encoder = AutoModel.from_pretrained("SpanBERT/spanbert-base-cased").to(device)
tokenizer = AutoTokenizer.from_pretrained('bert-base-cased')
sampleText = "Hello, my name is Damien. It's a cool name I think!"
tokenized_text = tokenizer.encode_plus(sampleText, return_tensors='pt')
tokenized_text["input_ids"] = tokenized_text["input_ids"].squeeze()
tokenized_text["token_type_ids"] = tokenized_text["token_type_ids"].squeeze()
tokenized_text["attention_mask"] = tokenized_text["attention_mask"].squeeze()
encoded_text = utils.spanbert_encode(tokenized_text, encoder)

extractor = mentions_extractor.MentionsExtractor(768, 0.25, 10).to(device)
linker = mentions_linker.MentionsLinker(extractor, 768).to(device)

res = linker(encoded_text)
print(res)


Some weights of BertModel were not initialized from the model checkpoint at SpanBERT/spanbert-base-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


(tensor([ 8, 11, 11, 11], device='cuda:0'), tensor([17, 13, 17, 15], device='cuda:0'), tensor([[  -inf,   -inf,   -inf,   -inf, 0.0000],
        [0.0956,   -inf,   -inf,   -inf, 0.0000],
        [0.0905, 0.0893,   -inf,   -inf, 0.0000],
        [0.0995, 0.0926, 0.0957,   -inf, 0.0000]], device='cuda:0',
       grad_fn=<CatBackward0>), tensor([[0, 0, 0, 0],
        [0, 0, 0, 0],
        [1, 0, 0, 0],
        [2, 1, 0, 0]], device='cuda:0'), tensor([[False, False, False, False],
        [ True, False, False, False],
        [ True,  True, False, False],
        [ True,  True,  True, False]], device='cuda:0'))


Get labels version 1:

Get labels version 2:

In [ ]:
import torch

cand_start = torch.tensor([0, 0, 1, 3, 5, 0])
cand_end = torch.tensor([0, 1, 3, 5, 6, 0])
true_start = torch.tensor([0, 0, 1, 3, 5])
true_end = torch.tensor([0, 1, 3, 5, 6])
labels = torch.tensor([2, 1, 2, 1, 1])
antecedents = torch.tensor([[0, 0, 0, 0, 0, 0],
                            [0, 0, 0, 0, 0, 0],
                            [1, 0, 0, 0, 0, 0],
                            [2, 1, 0, 0, 0, 0],
                            [3, 2, 1, 0, 0, 0],
                            [4, 3, 2, 1, 0, 0]])
antecedents_mask = torch.tensor([[False, False, False, False, False, False],
                                 [True, False, False, False, False, False],
                                 [True, True, False, False, False, False],
                                 [True, True, True, False, False, False],
                                 [True, True, True, True, False, False],
                                 [True, True, True, True, True, False]])

pairs_score = torch.rand([6, 6]) + torch.log(antecedents_mask)
print(pairs_score)

cand_indices = torch.stack([cand_start, cand_end], dim=1)
true_indices = torch.stack([true_start, true_end], dim=1)

predicted_expand = cand_indices.unsqueeze(1).expand(-1, true_indices.shape[0], cand_indices.shape[1])
print(predicted_expand.size())
true_expand = true_indices.unsqueeze(0).expand(cand_indices.shape[0], -1, cand_indices.shape[1])
print(true_expand.size())

same_span = (predicted_expand == true_expand).all(-1)
print(same_span.size())
candidate_labels = torch.matmul(same_span.float(), torch.unsqueeze(labels, 1).float()).squeeze().long()

print(candidate_labels)

antecedents_cluster = candidate_labels[antecedents] + torch.log(antecedents_mask.float())

print(antecedents_cluster)

antecedent_same_cluster = torch.logical_and(torch.eq(antecedents_cluster, candidate_labels.unsqueeze(1)),
                                            antecedents_cluster > 0)

print(antecedent_same_cluster)

has_dummy_antecedent = torch.logical_not(antecedent_same_cluster.any(dim=1, keepdim=True))

antecedents_labels = torch.cat([antecedent_same_cluster, has_dummy_antecedent], 1).float()

all_pairs_score = torch.cat([pairs_score, torch.zeros([6, 1])], 1)
true_pairs_score = all_pairs_score + torch.log(antecedents_labels)
print(antecedents_labels)
print(all_pairs_score)
print(true_pairs_score)

log_sum_true_antecedents_score = torch.logsumexp(true_pairs_score, 1)  # [k]
log_sum_all_antecedents_score = torch.logsumexp(all_pairs_score, 1)  # [k]
print(log_sum_true_antecedents_score)
print(log_sum_all_antecedents_score)
loss = -torch.sum(log_sum_true_antecedents_score - log_sum_all_antecedents_score)
print(loss)
print(loss.size())










Clusterize:

In [13]:
import re
import os
import config
from collections import defaultdict

doc_id = '11_alices_adventures_in_wonderland_brat.conll'
span_starts = [0, 3, 4]
span_ends = [3, 5, 4]
clusters = [0, 1, 0]

single_word = defaultdict(lambda: [])
starts = defaultdict(lambda: [])
ends = defaultdict(lambda: [])
for start, end, cluster_num in zip(span_starts, span_ends, clusters):
    if start == end:
        single_word[start].append(cluster_num)
    else:
        starts[start].append(cluster_num)
        ends[end].append(cluster_num)

os.makedirs(config.CONLL_EVAL_OUTPUT_FOLDER, exist_ok=True)
with open(config.CONLL_LITBANK_FOLDER + doc_id, mode='r', encoding='utf-8') as litbank_file, \
        open(config.CONLL_EVAL_OUTPUT_FOLDER + doc_id, mode='w', encoding='utf-8') as output_file:
    word_id = 0
    for line in litbank_file:
        # Start or end of doc, or blank line
        if line.startswith('#begin document') or line.startswith('#end document') or line == '\n':
            output_file.write(line)
            continue
        # "Word" line
        else:
            coref_indications = []
            for cluster_num in starts[word_id]:
                coref_indications.append(f"({cluster_num}")
            for cluster_num in single_word[word_id]:
                coref_indications.append(f"({cluster_num})")
            for cluster_num in ends[word_id]:
                coref_indications.append(f"{cluster_num})")
            coref_indication = "|".join(coref_indications) if len(coref_indications) != 0 else "_"

            parts = re.split("\s+", line.rstrip())
            parts[-1] = coref_indication
            output_line = "\t".join(parts)
            output_file.write(output_line + '\n')

            word_id += 1

Clusterize:

In [41]:
import torch
import networkx as nx

antecedents_mask = torch.tensor([[False, False, False, False, False, False],
                                 [True, False, False, False, False, False],
                                 [True, True, False, False, False, False],
                                 [True, True, True, False, False, False],
                                 [True, True, True, True, False, False],
                                 [True, True, True, True, True, False]])

pairs_score = torch.rand([6, 6]) + torch.log(antecedents_mask)
pairs_score = torch.cat([pairs_score, torch.tensor([0.0] * 6).unsqueeze(1)], 1)
k = pairs_score.size(0)
mask = torch.log(torch.cat([antecedents_mask, torch.tensor([True] * k).unsqueeze(1)], 1).float())
print(pairs_score)
best_antecedent = torch.argmax(pairs_score, 1)  # [k]
print(best_antecedent)
best_antecedent_pair = torch.stack([torch.tensor(range(k)), best_antecedent], 1).numpy()  # [k, 2]
print(best_antecedent_pair)
best_antecedent_pair = best_antecedent_pair[best_antecedent_pair[:, 1] != k]  # [num_spans_in_clusters, 2]
print(best_antecedent_pair)

clusters = []
G = nx.Graph()
G.add_edges_from(best_antecedent_pair)
for connected_nodes in nx.connected_components(G):
    clusters.append(list(connected_nodes))

print(clusters)

tensor([[  -inf,   -inf,   -inf,   -inf,   -inf,   -inf, 0.0000],
        [0.1641,   -inf,   -inf,   -inf,   -inf,   -inf, 0.0000],
        [0.5543, 0.0957,   -inf,   -inf,   -inf,   -inf, 0.0000],
        [0.4657, 0.0309, 0.3732,   -inf,   -inf,   -inf, 0.0000],
        [0.2609, 0.1000, 0.4695, 0.8064,   -inf,   -inf, 0.0000],
        [0.3100, 0.6399, 0.4986, 0.7278, 0.1717,   -inf, 0.0000]])
tensor([6, 0, 1, 6, 3, 3])
[[0 6]
 [1 0]
 [2 1]
 [3 6]
 [4 3]
 [5 3]]
[[1 0]
 [2 1]
 [4 3]
 [5 3]]
[[0, 1, 2], [3, 4, 5]]


In [35]:
import torch
t = torch.tensor([1.0] * 5).unsqueeze(1)
print(t)

tensor([[1.],
        [1.],
        [1.],
        [1.],
        [1.]])
